# Homework 6 - Decision Trees and Ensemble Learning


In [1]:
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.feature_extraction import DictVectorizer
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

target = 'fuel_efficiency_mpg'
df = pd.read_csv('car_fuel_efficiency_2026.csv').fillna(0)
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Preparation



In [2]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

y_train = df_train.pop(target)
y_val = df_val.pop(target)

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))

print(len(df_train), len(df_val), len(df_test))

6000 2000 2000


## Q1. Decision tree with max_depth=1


In [3]:
tree = DecisionTreeRegressor(max_depth=1, random_state=1)
tree.fit(X_train, y_train)

one_hot_feature = dv.feature_names_[tree.tree_.feature[0]]
original_feature = one_hot_feature.split('=', 1)[0]

print('One-hot feature:', one_hot_feature)
print('-> Original feature:', original_feature)

One-hot feature: model_year
-> Original feature: model_year


## Q2. Random forest with n_estimators=10


In [4]:
forest = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
forest.fit(X_train, y_train)

rmse_10 = mean_squared_error(y_val, forest.predict(X_val)) ** 0.5
print('-> RMSE:', round(rmse_10, 3))

-> RMSE: 1.833


## Q3. Tuning n_estimators



In [5]:
scores = {}
for n in [10, 50, 100, 150]:
    f = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    f.fit(X_train, y_train)
    scores[n] = mean_squared_error(y_val, f.predict(X_val)) ** 0.5
    print(f'n_estimators = {n:<4} RMSE = {scores[n]:.3f}')

print('-> Mejor n_estimators:', min(scores, key=scores.get))

n_estimators = 10   RMSE = 1.833


n_estimators = 50   RMSE = 1.774


n_estimators = 100  RMSE = 1.770


n_estimators = 150  RMSE = 1.769
-> Mejor n_estimators: 150


## Q4. Best max_depth


In [6]:
depth_scores = {}
for depth in [10, 15, 20, 25]:
    values = []
    for n in [10, 50, 100, 150]:
        f = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        f.fit(X_train, y_train)
        values.append(mean_squared_error(y_val, f.predict(X_val)) ** 0.5)
    depth_scores[depth] = np.mean(values)
    print(f'max_depth = {depth:<3} mean RMSE = {np.mean(values):.3f}')

print('-> Mejor max_depth:', min(depth_scores, key=depth_scores.get))

max_depth = 10  mean RMSE = 1.754


max_depth = 15  mean RMSE = 1.784


max_depth = 20  mean RMSE = 1.787


max_depth = 25  mean RMSE = 1.786
-> Mejor max_depth: 10


## Q5. Feature importance



In [7]:
imp = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
imp.fit(X_train, y_train)

importances = dict(zip(dv.feature_names_, imp.feature_importances_))
for feat in ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']:
    print(f'{feat:>20}: {importances.get(feat, 0.0):.4f}')
print('\n-> Mas importante:', max(['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement'], key=lambda f: importances.get(f, 0.0)))

      vehicle_weight: 0.2033
          horsepower: 0.0783
        acceleration: 0.0550
 engine_displacement: 0.0613

-> Mas importante: vehicle_weight


## Q6. XGBoost: tuning eta



In [8]:
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
watchlist = [(dtrain, 'train'), (dval, 'val')]

def xgb_rmse(eta):
    params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 0,
    }
    model = xgb.train(params, dtrain, num_boost_round=100, evals=watchlist, verbose_eval=False)
    return mean_squared_error(y_val, model.predict(dval)) ** 0.5

rmse_03 = xgb_rmse(0.3)
rmse_01 = xgb_rmse(0.1)
print('eta = 0.3 ->', round(rmse_03, 3))
print('eta = 0.1 ->', round(rmse_01, 3))
print('-> Mejor eta: 0.1')

eta = 0.3 -> 1.826
eta = 0.1 -> 1.725
-> Mejor eta: 0.1
